## Creates new transactions table 
### has two additional columns: elus and koht (with values YES/'')

In [1]:
import sqlite3
import pandas as pd
import sys
sys.path.append("..")
from common_display import display_db_table 

## Configuration

In [2]:
DB_DIR = "../example_data"

# transaktsioonide andmebaas
TRANSACTION_DB = f"{DB_DIR}/transactions.db"
# elus/koht nimekirjade db
PHRASE_PATTERNS = f"{DB_DIR}/phrase_patterns.db"
# tabelite nimed elusolendite ja kohtade jaoks
elustabel = 'elus_v1'
kohttabel = 'koht_v1'

# enriched transactions
ENRICHED_TRANSACTIONS = f"{DB_DIR}/enriched_transactions.db"
# uus enriched transaktsioonide tabel
TRANSACTION_ROW = "transaction_row"

## Connect to db

In [11]:
con = sqlite3.connect(ENRICHED_TRANSACTIONS)
cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{TRANSACTION_DB}" AS trans')
cur.execute(f'ATTACH DATABASE "{PHRASE_PATTERNS}" AS maarused')

## Workflow

## Create new transaction table

### juurde lisada veerud koht ja elus, kus on listide põhjal otsus sõna kohta

Running the drop command will drop the trans.transaction_row table. 

In [ ]:
#cur.execute("""DROP TABLE if exists {new_table}""".format(new_table = TRANSACTION_ROW))

In [15]:
%%time

cur.execute("""CREATE TABLE {new_table} as SELECT * FROM trans.transaction_row """.format(new_table = TRANSACTION_ROW))

CPU times: user 5.71 ms, sys: 321 µs, total: 6.04 ms
Wall time: 10.9 ms


In [17]:
# add default values for koht and elus
cur.execute("""
ALTER TABLE {new_table}
ADD koht VARCHAR(50) DEFAULT ''
""".format(new_table = TRANSACTION_ROW))
con.commit()

cur.execute("""
ALTER TABLE {new_table}
ADD elus VARCHAR(50) DEFAULT ''
""".format(new_table = TRANSACTION_ROW))
con.commit()

In [18]:
%%time

cur.execute("""
UPDATE {new_table}
SET elus = 'YES'
WHERE lower({new_table2}.lemma) in (select lower(lemma) from maarused.{elustbl})
""".format(new_table = TRANSACTION_ROW, new_table2 = TRANSACTION_ROW, elustbl=elustabel))

con.commit()

CPU times: user 7.49 ms, sys: 3.22 ms, total: 10.7 ms
Wall time: 14.2 ms


In [19]:
%%time

cur.execute("""
UPDATE {new_table}
SET koht = 'YES'
WHERE lower({new_table2}.lemma) in (select lower(lemma) from maarused.{kohttbl})
""".format(new_table = TRANSACTION_ROW, new_table2 = TRANSACTION_ROW, kohttbl=kohttabel))

con.commit()

CPU times: user 6.38 ms, sys: 1.17 ms, total: 7.55 ms
Wall time: 10.5 ms


In [21]:
display_db_table(con, TRANSACTION_ROW)

,id,head_id,loc,loc_rel,deprel,form,lemma,feats,parent_loc,pos,koht,elus
0,1,2,3,-1,obl,lõpus,lõpp,"com,in,sg",None,S,,
1,2,2,5,1,nsubj,Türi,Türi,"gen,prop,sg",None,S,,
2,3,2,6,2,obl,1.,1.,"<?>,ord,roman",None,N,,
3,4,3,1,-3,obj,Bändi,bänd,"adit,com,sg",None,S,,YES
4,5,3,9,-2,nsubj,kidramees,kidramees,"com,nom,sg",None,S,,


In [25]:
con.close()